# 부산 침수 예측 · 강화학습 경로 모델링

## Goal

이 공개용 워크스루는 원본 팀 프로젝트의 모델링 흐름을 검토하기 위한 정리본입니다. 침수 예측 모델과 강화학습 모델링을 수행했습니다.

원본 데이터와 실행 결과는 포함하지 않습니다. 승인된 비공개 입력이 있을 때만 아래 단계를 실행하세요.

## Setup

필수 패키지는 `requirements.txt`에 있습니다. 이 노트북은 설치 명령을 실행하지 않으며, 데이터 파일이 없으면 의도적으로 중단됩니다.

In [ ]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Final, TypeAlias

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

NodeId: TypeAlias = str | int
RANDOM_STATE: Final = 42
PRIVATE_DATA_DIR: Final = Path('..') / 'data' / 'private'
FEATURE_COLUMNS: Final = (
    'elevation', 'slope', 'storage_distance_m', 'storage_capacity',
    'pump_distance_m', 'pump_capacity', 'hourly_rainfall',
)


## 1. Validate private inputs

`flood_features.csv`에는 `flood_label`과 위 특성 열이, `road_edges.csv`에는 `from_node`, `to_node`, `length_m`, `is_flooded`가 필요합니다. 실제 값·좌표·식별자는 이 저장소에 넣지 않습니다.

In [ ]:
def require_columns(frame: pd.DataFrame, required: Sequence[str], name: str) -> None:
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f'{name} is missing required columns: {missing}')

flood_path = PRIVATE_DATA_DIR / 'flood_features.csv'
road_path = PRIVATE_DATA_DIR / 'road_edges.csv'
if not flood_path.exists() or not road_path.exists():
    raise FileNotFoundError(
        'Approved private inputs are required. See docs/data-sources.md and docs/usage.md.'
    )

flood_features = pd.read_csv(flood_path)
road_edges = pd.read_csv(road_path)
require_columns(flood_features, (*FEATURE_COLUMNS, 'flood_label'), 'flood_features.csv')
require_columns(road_edges, ['from_node', 'to_node', 'length_m', 'is_flooded'], 'road_edges.csv')


## 2. Train a flood classifier

학습/검증 분할 후 학습 데이터에만 스케일링과 SMOTE를 적용합니다. 이는 불균형 라벨을 다루되 검증 데이터가 학습 과정에 섞이지 않도록 하기 위한 순서입니다.

In [ ]:
X = flood_features.loc[:, list(FEATURE_COLUMNS)].copy()
y = flood_features['flood_label'].astype(int)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_train_balanced, y_train_balanced = SMOTE(random_state=RANDOM_STATE).fit_resample(
    X_train_scaled, y_train
)

flood_model = RandomForestClassifier(
    n_estimators=300, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1
)
flood_model.fit(X_train_balanced, y_train_balanced)
valid_prediction = flood_model.predict(X_valid_scaled)
print(classification_report(y_valid, valid_prediction, zero_division=0))


## 3. Q-learning route-policy example

도로 링크의 침수 상태와 길이를 보상에 반영하는 최소 예제입니다. 보상 계수와 종료 조건은 실제 운영 정책이 아니라 실험 가정이며, 별도 검증이 필요합니다.

In [ ]:
@dataclass(frozen=True, slots=True)
class RouteAction:
    destination: NodeId
    length_m: float
    is_flooded: bool

@dataclass(frozen=True, slots=True)
class QTrainingConfig:
    episodes: int = 500
    alpha: float = 0.1
    gamma: float = 0.95
    flood_penalty: float = 1_000.0
    seed: int = RANDOM_STATE

def build_actions(edges: pd.DataFrame) -> dict[NodeId, list[RouteAction]]:
    actions: dict[NodeId, list[RouteAction]] = {}
    for edge in edges.itertuples(index=False):
        actions.setdefault(edge.from_node, []).append(
            RouteAction(edge.to_node, float(edge.length_m), bool(edge.is_flooded))
        )
    return actions

def train_q_policy(
    actions: Mapping[NodeId, Sequence[RouteAction]], start: NodeId, goal: NodeId,
    *, config: QTrainingConfig = QTrainingConfig(),
) -> Mapping[tuple[NodeId, int], float]:
    rng = np.random.default_rng(config.seed)
    q_values = {(state, action): 0.0 for state, moves in actions.items() for action in range(len(moves))}
    for episode in range(config.episodes):
        state, epsilon, steps = start, max(0.05, 1.0 - episode / config.episodes), 0
        while state != goal and state in actions and steps < 500:
            action_count = len(actions[state])
            action = int(rng.integers(action_count)) if rng.random() < epsilon else max(
                range(action_count), key=lambda index: q_values[(state, index)]
            )
            route_action = actions[state][action]
            next_state = route_action.destination
            reward = -route_action.length_m - (config.flood_penalty if route_action.is_flooded else 0.0)
            if next_state == goal:
                reward += config.flood_penalty
            future = max((q_values[(next_state, index)] for index in range(len(actions.get(next_state, [])))), default=0.0)
            q_values[(state, action)] += config.alpha * (reward + config.gamma * future - q_values[(state, action)])
            state, steps = next_state, steps + 1
    return q_values

actions = build_actions(road_edges)
# Select start and goal using an approved experiment configuration; do not hard-code source identifiers.
print(f'Prepared {len(actions)} route states for a private experiment.')


## Checks and next steps

이 노트북에는 공개 가능한 입력·출력이 없으므로 성능이나 경로 품질을 해석하지 않습니다. 적법한 데이터로 실행할 경우에는 데이터 분할, 불균형 처리, 보상 가정, 안전성 검토를 기록하고 결과를 재공개하기 전에 별도의 권한 검토를 수행하세요.